## NB_patient_360
Builds **`<catalog>.gold.patient_360`** as defined in `Gold/sttm_silver_to_gold.xlsx` (sheet `patient_360`).

* **Grain:** one row per patient
* **Driving table:** `<catalog>.silver.patients`
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id = hospital_id`, `silver.insurance_providers` on `insurance_provider_id`, aggregated `silver.lab_results` grouped by `patient_id`
* **Filters:** none, all patients included
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
# Catalog placeholder from the STTM (<catalog>); defaults to adb_caresync
catalog = "adb_caresync"

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"

print("=" * 80)
print("[STEP 1] Notebook parameters")
print(f"  catalog       : {catalog}")
print(f"  source schema : {source_schema}")
print(f"  target table  : {target_table}")
print("=" * 80)

In [0]:
# Lab results are pre-aggregated per patient so the join keeps one row per patient
patient_360_sql = f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                                AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)  AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)  AS critical_test_count,
    MAX(test_date)                                          AS last_test_date
  FROM {source_schema}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                   AS patient_id,
  CAST(p.first_name AS STRING)                                                AS first_name,
  CAST(p.last_name AS STRING)                                                 AS last_name,
  CAST(p.date_of_birth AS DATE)                                               AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                                    AS gender,
  CAST(p.city AS STRING)                                                      AS city,
  CAST(p.state AS STRING)                                                     AS state,
  CAST(p.registered_hospital_id AS INT)                                       AS hospital_id,
  CAST(h.hospital_name AS STRING)                                             AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                        AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                            AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                                    AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                            AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                            AS critical_test_count,
  CAST(la.last_test_date AS DATE)                                             AS last_test_date
FROM {source_schema}.patients p
LEFT JOIN {source_schema}.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {source_schema}.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id
"""

print(f"[STEP 2] Building {target_table}")
print("  Query prepared:")
print(patient_360_sql)
spark.sql(patient_360_sql)
print(f"  {target_table} created successfully")
print("=" * 80)

In [0]:
# Validate grain (one row per patient) against the driving table
validation_sql = f"""
SELECT
  (SELECT COUNT(*) FROM {source_schema}.patients)          AS source_patients,
  COUNT(*)                                                 AS target_rows,
  COUNT(DISTINCT patient_id)                               AS distinct_patients,
  SUM(total_tests)                                         AS total_tests,
  (SELECT COUNT(*) FROM {source_schema}.lab_results)       AS source_lab_results,
  COUNT(CASE WHEN hospital_name IS NULL THEN 1 END)        AS missing_hospital_name,
  COUNT(CASE WHEN insurance_provider_name IS NULL THEN 1 END) AS missing_insurance_name
FROM {target_table}
"""
print("[STEP 3] Validating")
print("  Query prepared:")
print(validation_sql)
v = spark.sql(validation_sql).first()
for k, val in v.asDict().items():
    print(f"  {k:<24}: {val}")

if not (v.source_patients == v.target_rows == v.distinct_patients):
    raise ValueError("Grain check failed: patient_360 must have exactly one row per source patient")
print("  Grain check passed: one row per patient")
print("=" * 80)

display(spark.table(target_table).orderBy("patient_id").limit(10))

In [0]:
# Return the gold table record count to the caller (ADF / parent notebook)
record_count = spark.table(target_table).count()
print(f"[STEP 4] Record count for {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))